# Phase 7: Model Fine-tuning

In [8]:
import pandas as pd                                                                                                     
import torch                                                                                                            
from torch.utils.data import Dataset, DataLoader                                                                        
from transformers import AutoTokenizer                                                                                  
import gc                                                                                                               
import warnings                                                                                                         
warnings.filterwarnings('ignore')   

import matplotlib.pyplot as plt                                                                                         
from matplotlib.backends.backend_pdf import PdfPages                                                                    
from sklearn.metrics import classification_report                                                                       
import pandas as pd                                                                                                     
from statsmodels.stats.contingency_tables import mcnemar  

import nltk                                                                                                                                                         
from nltk.sentiment.vader import SentimentIntensityAnalyzer                                                                                                         
from sklearn.feature_extraction.text import TfidfVectorizer                                                                                                         
from sklearn.svm import SVC     
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')                                                                                               
print(f"Using device: {device}")  

Using device: cuda


## Step 7.1 Setup, Label Encoding & Tokenization

In [7]:
# 1. Load Data                                                                                                                                                      
train_df = pd.read_csv('Augmented_Ground_Truth_714.csv')                                                         
val_df = pd.read_csv('validation_set_400_final.csv')                                                        
test_df = pd.read_csv('test_set_400_final.csv')                                                                                                           
                                                
# 2. Label Encoding (Strictly maps to 0, 1, 2 regardless of CSV casing)                                                                                             
label2id = {'POSITIVE': 0, 'NEUTRAL': 1, 'NEGATIVE': 2}                                                                                                             
id2label = {0: 'POSITIVE', 1: 'NEUTRAL', 2: 'NEGATIVE'}                                                                                                             
                                                                                                                                                                    
train_df['encoded_label'] = train_df['sentiment'].astype(str).str.upper().map(label2id)                                                                             
val_df['encoded_label'] = val_df['sentiment'].astype(str).str.upper().map(label2id)                                                                                 
                                                                                                                                                                    
# 3. XLM-RoBERTa Tokenization & PyTorch Dataset                                                                                                                     
MODEL_NAME = 'xlm-roberta-base'                                                                                                                                     
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)                                                                                                               
                                                                                                                                                                    
class ABSADataset(Dataset):                                                                                                                                         
    def __init__(self, df, tokenizer, max_length=128):                                                                                                              
        self.encodings = tokenizer(                                                                                                                                 
            df['TargetAspect'].tolist(),                                                                                                                            
            df['sentence'].tolist(),                                                                                                                                
            truncation=True,                                                                                                                                        
            padding=True,                                                                                                                                           
            max_length=max_length,                                                                                                                                  
            return_tensors='pt'                                                                                                                                     
        )                                                                                                                                                           
        self.labels = torch.tensor(df['encoded_label'].tolist(), dtype=torch.long)                                                                                  
                                                                                                                                                                    
    def __len__(self):                                                                                                                                              
        return len(self.labels)                                                                                                                                     
                                                                                                                                                                    
    def __getitem__(self, idx):                                                                                                                                     
        item = {key: val[idx] for key, val in self.encodings.items()}                                                                                               
        item['labels'] = self.labels[idx]                                                                                                                           
        return item                                                                                                                                                 
                                                                                                                                                                    
train_dataset = ABSADataset(train_df, tokenizer)                                                                                                                    
val_dataset = ABSADataset(val_df, tokenizer)                                                                                                                        
print("Datasets loaded into memory successfully.")                                                                                                                          

Datasets loaded into memory successfully.


In [10]:
# 1. VADER Lexicon                                                                                                                                                  
nltk.download('vader_lexicon', quiet=True)                                                                                                                          
sia = SentimentIntensityAnalyzer()                                                                                                                                  
def predict_vader(text):                                                                                                                                            
    score = sia.polarity_scores(str(text))['compound']                                                                                                              
    if score >= 0.05: return 0      # Positive                                                                                                                      
    elif score <= -0.05: return 2   # Negative                                                                                                                      
    else: return 1                  # Neutral                                                                                                                       
vader_preds = val_df['sentence'].apply(predict_vader).tolist()                                                                                                      
                                                                                                                                                                    
# 2. SVM with TF-IDF                                                                                                                                                
vectorizer = TfidfVectorizer(max_features=5000)                                                                                                                     
X_train_svm = vectorizer.fit_transform(train_df['sentence'])                                                                                                        
X_val_svm = vectorizer.transform(val_df['sentence'])                                                                                                                
                                                                                                                                                                    
svm_model = SVC(kernel='linear')                                                                                                                                    
svm_model.fit(X_train_svm, train_df['encoded_label'])                                                                                                               
svm_preds = svm_model.predict(X_val_svm).tolist()

## (Freezing + Warmup + Smoothing)

In [12]:
from transformers import XLMRobertaForSequenceClassification, get_linear_schedule_with_warmup                                                                       
from torch.optim import AdamW                                                                                                                                       
from torch.nn import CrossEntropyLoss                                                                                                                               
from sklearn.metrics import f1_score                                                                                                                                
import matplotlib.pyplot as plt                                                                                                                                     
import gc                  

In [13]:
OPTIMAL_BS = 32                                                                                                                                                     
LR = 3e-5                                                                                                                                                           
EPOCHS = 5                                                                                                                                                          
WEIGHT_DECAY = 0.1                                                                                                                                                
LABEL_SMOOTHING = 0.1                                                                                                                                               
                        